In [ ]:
import os
import numpy as np
import pandas as pd
import torch

from scipy.sparse import coo_matrix, csr_matrix, diags
from scipy.sparse.csgraph import dijkstra
from torch_geometric.data import Data

In [ ]:
# ==============================
# -2. Input/output paths
# Set environment variables to override these defaults.
# ==============================
PROJECT_ROOT = os.environ.get("DEEPGLOC_PROJECT_ROOT", os.getcwd())
AUGMENTED_GRAPH_DIR = os.environ.get(
    "DEEPGLOC_AUGMENTED_GRAPH_DIR",
    os.path.join(PROJECT_ROOT, "augmented_graph_output")
)
CORE_DIR = os.environ.get("DEEPGLOC_CORE_DIR", os.path.join(PROJECT_ROOT, "core_gene"))

AUG_EDGE_FILE = os.environ.get(
    "DEEPGLOC_AUG_EDGE_FILE",
    os.path.join(AUGMENTED_GRAPH_DIR, "augmented_graph_edge_list.tsv")
)
NODE_FILE = os.environ.get(
    "DEEPGLOC_NODE_FILE",
    os.path.join(AUGMENTED_GRAPH_DIR, "augmented_graph_node_table.tsv")
)
OUT_PYG_FILE = os.environ.get(
    "DEEPGLOC_OUT_PYG_FILE",
    os.path.join(AUGMENTED_GRAPH_DIR, "pyg_data_with_topology_features.pt")
)


In [ ]:
# ==============================
# -3. 读取边表和节点表
# ==============================
edge_df = pd.read_csv(AUG_EDGE_FILE, sep="\t")
node_df = pd.read_csv(NODE_FILE, sep="\t")

print("edge_df shape:", edge_df.shape)
print("node_df shape:", node_df.shape)


In [ ]:
# ==============================
# -4. 四个过程标签映射
# ==============================
process_map = {
    "Biosynthesis": 0,
    "Esterification": 1,
    "Excretion": 2,
    "Uptake": 3
}

core_files = {
    "Biosynthesis": "core_gene_Biosynthesis.csv",
    "Esterification": "core_gene_Esterification.csv",
    "Excretion": "core_gene_Excretion.csv",
    "Uptake": "core_gene_Uptake.csv",
}


In [ ]:
# ==============================
# -5. 读取 core gene 文件，构建 core_all
# ==============================
core_list = []
for proc, fn in core_files.items():
    fpath = os.path.join(CORE_DIR, fn)
    df = pd.read_csv(fpath)

    # 只取前两列：ENSGID, SYMBOL
    df = df.iloc[:, :2].copy()
    df.columns = ["ENSGID", "SYMBOL"]

    df["ENSGID"] = df["ENSGID"].astype(str).str.strip().str.split(".").str[0]
    df["SYMBOL"] = df["SYMBOL"].astype(str).str.strip()
    df["process"] = proc
    df["label"] = process_map[proc]

    core_list.append(df)

core_all = pd.concat(core_list, axis=0, ignore_index=True)

print("core_all shape:", core_all.shape)
print("core unique genes:", core_all["ENSGID"].nunique())


In [ ]:
# ==============================
# -6. 区分 hard seed 和 ambiguous seed
# ==============================
core_count = core_all.groupby("ENSGID")["process"].nunique().reset_index()
core_count.columns = ["ENSGID", "n_process"]

core_all2 = core_all.merge(core_count, on="ENSGID", how="left")

hard_seed_df = core_all2[core_all2["n_process"] == 1].copy()
ambiguous_seed_df = core_all2[core_all2["n_process"] > 1].copy()

print("hard seeds unique genes:", hard_seed_df["ENSGID"].nunique())
print("ambiguous seeds unique genes:", ambiguous_seed_df["ENSGID"].nunique())


In [ ]:
# ==============================
# -7. 统一 node_df / edge_df / core gene 的 ENSGID 格式
# ==============================
node_df = node_df.copy()
edge_df = edge_df.copy()

node_df["ENSGID"] = node_df["ENSGID"].astype(str).str.strip().str.split(".").str[0]
edge_df["gene1"] = edge_df["gene1"].astype(str).str.strip().str.split(".").str[0]
edge_df["gene2"] = edge_df["gene2"].astype(str).str.strip().str.split(".").str[0]

core_all["ENSGID"] = core_all["ENSGID"].astype(str).str.strip().str.split(".").str[0]
hard_seed_df["ENSGID"] = hard_seed_df["ENSGID"].astype(str).str.strip().str.split(".").str[0]
ambiguous_seed_df["ENSGID"] = ambiguous_seed_df["ENSGID"].astype(str).str.strip().str.split(".").str[0]


In [ ]:
# ==============================
# -8. 节点排序并建立 gene2idx / idx2gene
# ==============================
node_df = node_df.sort_values("ENSGID").reset_index(drop=True)

gene2idx = {g: i for i, g in enumerate(node_df["ENSGID"])}
idx2gene = {i: g for g, i in gene2idx.items()}

num_nodes = len(node_df)

print("num_nodes:", num_nodes)


In [ ]:
# ==============================
# -9. 简单检查
# ==============================
print("\nColumns in node_df:")
print(node_df.columns.tolist())

print("\nColumns in edge_df:")
print(edge_df.columns.tolist())

print("\nExample node_df head:")
display(node_df.head())

print("\nExample edge_df head:")
display(edge_df.head())

In [ ]:
# ==============================
# 0. 基础清理
# ==============================
EPS = 1e-8
RWR_ALPHA = 0.15
RWR_MAX_ITER = 300
RWR_TOL = 1e-10

node_df = node_df.copy()
edge_df = edge_df.copy()
core_all = core_all.copy()
hard_seed_df = hard_seed_df.copy()
ambiguous_seed_df = ambiguous_seed_df.copy()

node_df["ENSGID"] = node_df["ENSGID"].astype(str).str.strip().str.split(".").str[0]
edge_df["gene1"] = edge_df["gene1"].astype(str).str.strip().str.split(".").str[0]
edge_df["gene2"] = edge_df["gene2"].astype(str).str.strip().str.split(".").str[0]
core_all["ENSGID"] = core_all["ENSGID"].astype(str).str.strip().str.split(".").str[0]
hard_seed_df["ENSGID"] = hard_seed_df["ENSGID"].astype(str).str.strip().str.split(".").str[0]
ambiguous_seed_df["ENSGID"] = ambiguous_seed_df["ENSGID"].astype(str).str.strip().str.split(".").str[0]

node_df["is_in_meta"] = pd.to_numeric(node_df["is_in_meta"], errors="coerce").fillna(0).astype(int)
node_df["core_type"] = node_df["core_type"].fillna("").astype(str)

node_df["is_native_core"] = node_df["core_type"].str.contains("native_core").astype(int)
node_df["is_anchored_core"] = node_df["core_type"].str.contains("anchored_core").astype(int)


def zscore_safe(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if (not np.isfinite(sd)) or sd < 1e-12:
        return np.zeros_like(x, dtype=float)
    out = (x - mu) / sd
    out[~np.isfinite(out)] = 0.0
    return out


In [ ]:
# ==============================
# 1. 只保留图中存在的边
# ==============================
edge_df = edge_df[
    edge_df["gene1"].isin(gene2idx) &
    edge_df["gene2"].isin(gene2idx)
].copy()

edge_df["edge_weight"] = pd.to_numeric(edge_df["edge_weight"], errors="coerce").fillna(0.0)
edge_df = edge_df[edge_df["edge_weight"] > 0].copy()

edge_type_map = {
    "meta": 0,
    "ppi_anchor": 1
}
edge_df["edge_type_id"] = edge_df["edge_type"].map(edge_type_map).fillna(-1).astype(int)

src = edge_df["gene1"].map(gene2idx).to_numpy(dtype=int)
dst = edge_df["gene2"].map(gene2idx).to_numpy(dtype=int)
w = edge_df["edge_weight"].to_numpy(dtype=float)
t = edge_df["edge_type_id"].to_numpy(dtype=int)

print("filtered undirected edges:", edge_df.shape[0])
print("num_nodes:", num_nodes)


In [ ]:
# ==============================
# 2. 建立稀疏图
#    A_all: 最终综合图
#    A_meta: 仅 meta 边
#    A_ppi: 仅 ppi_anchor 边
# ==============================
def build_undirected_sparse(src, dst, weight, n_nodes):
    row = np.concatenate([src, dst])
    col = np.concatenate([dst, src])
    dat = np.concatenate([weight, weight])
    A = coo_matrix((dat, (row, col)), shape=(n_nodes, n_nodes)).tocsr()
    A.sum_duplicates()
    return A

A_all = build_undirected_sparse(src, dst, w, num_nodes)

meta_mask = (t == 0)
ppi_mask = (t == 1)

A_meta = build_undirected_sparse(src[meta_mask], dst[meta_mask], w[meta_mask], num_nodes)
A_ppi  = build_undirected_sparse(src[ppi_mask],  dst[ppi_mask],  w[ppi_mask],  num_nodes)

In [ ]:
# ==============================
# 3. 基础图特征
# ==============================
degree = np.diff(A_all.indptr).astype(float)
weighted_degree = np.asarray(A_all.sum(axis=1)).ravel().astype(float)

meta_degree = np.diff(A_meta.indptr).astype(float)
meta_strength = np.asarray(A_meta.sum(axis=1)).ravel().astype(float)

ppi_degree = np.diff(A_ppi.indptr).astype(float)
ppi_strength = np.asarray(A_ppi.sum(axis=1)).ravel().astype(float)

ppi_ratio = ppi_strength / (weighted_degree + EPS)
has_ppi_edge = (ppi_degree > 0).astype(int)

node_df["degree"] = degree
node_df["weighted_degree"] = weighted_degree

node_df["meta_degree"] = meta_degree
node_df["meta_strength"] = meta_strength

node_df["ppi_degree"] = ppi_degree
node_df["ppi_strength"] = ppi_strength
node_df["ppi_ratio"] = ppi_ratio
node_df["has_ppi_edge"] = has_ppi_edge

# 标准化版本（给 x 用）
node_df["degree_z"] = zscore_safe(node_df["degree"])
node_df["weighted_degree_z"] = zscore_safe(node_df["weighted_degree"])
node_df["meta_degree_z"] = zscore_safe(node_df["meta_degree"])
node_df["meta_strength_z"] = zscore_safe(node_df["meta_strength"])
node_df["ppi_degree_z"] = zscore_safe(node_df["ppi_degree"])
node_df["ppi_strength_z"] = zscore_safe(node_df["ppi_strength"])

In [ ]:
# ==============================
# 4. 四个过程的 seed 集合
#    这里用“全部 core genes”定义每个过程 seed，
#    包括重叠 core（如果某基因属于多个过程，它可以进入多个过程的 seed 集）
# ==============================
process_names = list(process_map.keys())

process_to_seed_genes = {}
process_to_seed_idx = {}

for p in process_names:
    genes = core_all.loc[core_all["process"] == p, "ENSGID"].dropna().astype(str).unique().tolist()
    idxs = sorted({gene2idx[g] for g in genes if g in gene2idx})
    process_to_seed_genes[p] = genes
    process_to_seed_idx[p] = np.array(idxs, dtype=int)
    print(f"{p}: {len(idxs)} seeds in graph")

In [ ]:
# ==============================
# 5. seed_weight_sum_4类
#    1-hop 直接连接到各过程 seed 的加权强度
# ==============================
for p in process_names:
    seed_idx = process_to_seed_idx[p]
    col_name = f"seed_weight_sum_{p}"
    if len(seed_idx) == 0:
        node_df[col_name] = 0.0
    else:
        vals = np.asarray(A_all[:, seed_idx].sum(axis=1)).ravel().astype(float)
        node_df[col_name] = vals

    node_df[f"{col_name}_z"] = zscore_safe(node_df[col_name].values)

In [ ]:
# ==============================
# 6. avg_dist_to_4类seed
#    用 1 / edge_weight 作为距离
# ==============================
dist_w = 1.0 / np.clip(w, EPS, None)
A_dist = build_undirected_sparse(src, dst, dist_w, num_nodes)

def avg_dist_to_seed_set(A_dist, seed_idx):
    if len(seed_idx) == 0:
        return np.full(A_dist.shape[0], np.nan, dtype=float)

    dist_mat = dijkstra(A_dist, directed=False, indices=seed_idx)
    dist_mat = np.atleast_2d(dist_mat)

    # 对每个节点，取到该 seed 集的平均最短路径
    dist_mat[~np.isfinite(dist_mat)] = np.nan
    avg_dist = np.nanmean(dist_mat, axis=0)

    # 不连通时填一个较大的值
    finite_mask = np.isfinite(avg_dist)
    if finite_mask.any():
        fill_value = np.nanmax(avg_dist[finite_mask]) + 1.0
    else:
        fill_value = 1e6
    avg_dist[~finite_mask] = fill_value
    return avg_dist.astype(float)

for p in process_names:
    seed_idx = process_to_seed_idx[p]
    col_name = f"avg_dist_{p}"
    node_df[col_name] = avg_dist_to_seed_set(A_dist, seed_idx)
    node_df[f"{col_name}_z"] = zscore_safe(node_df[col_name].values)

In [ ]:
# ==============================
# 7. diffusion_score_4类
#    Random Walk with Restart / Personalized diffusion
# ==============================
row_sum = np.asarray(A_all.sum(axis=1)).ravel().astype(float)
inv_row_sum = 1.0 / np.clip(row_sum, EPS, None)
P = diags(inv_row_sum) @ A_all   # row-normalized transition matrix

def rwr_score(P, seed_idx, alpha=0.15, max_iter=300, tol=1e-10):
    n = P.shape[0]
    if len(seed_idx) == 0:
        return np.zeros(n, dtype=float)

    p0 = np.zeros(n, dtype=float)
    p0[seed_idx] = 1.0 / len(seed_idx)

    p = p0.copy()
    for _ in range(max_iter):
        p_next = alpha * p0 + (1.0 - alpha) * (P.T @ p)
        p_next = np.asarray(p_next).ravel()

        if np.linalg.norm(p_next - p, ord=1) < tol:
            p = p_next
            break
        p = p_next

    return p.astype(float)

for p in process_names:
    seed_idx = process_to_seed_idx[p]
    col_name = f"diffusion_{p}"
    node_df[col_name] = rwr_score(
        P=P,
        seed_idx=seed_idx,
        alpha=RWR_ALPHA,
        max_iter=RWR_MAX_ITER,
        tol=RWR_TOL
    )
    node_df[f"{col_name}_z"] = zscore_safe(node_df[col_name].values)

In [ ]:
# ==============================
# 8. 构建标签 y / mask
#    不做 train/val/test 切分
# ==============================
y = torch.full((num_nodes,), -1, dtype=torch.long)

all_core_mask = torch.zeros(num_nodes, dtype=torch.bool)
hard_seed_mask = torch.zeros(num_nodes, dtype=torch.bool)
ambiguous_core_mask = torch.zeros(num_nodes, dtype=torch.bool)

seed_weight = torch.zeros(num_nodes, dtype=torch.float32)

# 全部 core
all_core_genes = set(core_all["ENSGID"].dropna().astype(str).tolist())
for g in all_core_genes:
    if g in gene2idx:
        all_core_mask[gene2idx[g]] = True

# ambiguous core
ambiguous_genes = set(ambiguous_seed_df["ENSGID"].dropna().astype(str).tolist())
for g in ambiguous_genes:
    if g in gene2idx:
        ambiguous_core_mask[gene2idx[g]] = True

# hard seed：单标签 core，用于监督
hard_seed_unique = hard_seed_df[["ENSGID", "label"]].drop_duplicates().copy()
hard_seed_map = dict(zip(hard_seed_unique["ENSGID"], hard_seed_unique["label"]))

for gene, label in hard_seed_map.items():
    if gene in gene2idx:
        idx = gene2idx[gene]
        y[idx] = int(label)
        hard_seed_mask[idx] = True

        core_type_val = str(node_df.loc[idx, "core_type"])
        if "native_core" in core_type_val:
            seed_weight[idx] = 1.0
        elif "anchored_core" in core_type_val:
            seed_weight[idx] = 0.6
        else:
            seed_weight[idx] = 1.0

# 这里不做 train/val/test
# supervised_mask 才是你真正训练交叉熵的节点
supervised_mask = hard_seed_mask.clone()

print("all core in graph:", int(all_core_mask.sum()))
print("hard supervised seeds:", int(supervised_mask.sum()))
print("ambiguous core in graph:", int(ambiguous_core_mask.sum()))

for p, lab in process_map.items():
    print(p, int(((y == lab) & supervised_mask).sum()))


In [ ]:
# ==============================
# 9. 构建新的 x
#    去掉 proc_* / is_core_gene，避免标签泄漏
# ==============================
feature_cols = [
    # 图基础特征
    "is_in_meta",
    "degree_z",
    "weighted_degree_z",

    # source-aware
    "meta_strength_z",
    "ppi_strength_z",
    "ppi_ratio",
    "has_ppi_edge",

    # 直接 seed 邻近
    "seed_weight_sum_Biosynthesis_z",
    "seed_weight_sum_Esterification_z",
    "seed_weight_sum_Excretion_z",
    "seed_weight_sum_Uptake_z",

    # 图距离
    "avg_dist_Biosynthesis_z",
    "avg_dist_Esterification_z",
    "avg_dist_Excretion_z",
    "avg_dist_Uptake_z",

    # diffusion / propagation
    "diffusion_Biosynthesis_z",
    "diffusion_Esterification_z",
    "diffusion_Excretion_z",
    "diffusion_Uptake_z",
]

# 若你想保留图构造角色信息，可取消下面这两列注释
# feature_cols += ["is_native_core", "is_anchored_core"]

x = torch.tensor(
    node_df[feature_cols].fillna(0.0).to_numpy(dtype=np.float32),
    dtype=torch.float32
)

print("x shape:", x.shape)
print("feature cols:", feature_cols)



In [ ]:
# ==============================
# 10. 构建 edge_index / edge_attr
#     这里仍然双向展开
#     edge_attr 改成: [edge_weight, is_meta_edge, is_ppi_edge]
# ==============================
edge_index = torch.tensor(
    np.vstack([
        np.concatenate([src, dst]),
        np.concatenate([dst, src])
    ]),
    dtype=torch.long
)

edge_weight = torch.tensor(np.concatenate([w, w]), dtype=torch.float32)

edge_is_meta = np.concatenate([(t == 0).astype(float), (t == 0).astype(float)])
edge_is_ppi  = np.concatenate([(t == 1).astype(float), (t == 1).astype(float)])

edge_attr = torch.tensor(
    np.vstack([
        np.concatenate([w, w]),
        edge_is_meta,
        edge_is_ppi
    ]).T,
    dtype=torch.float32
)

edge_type = torch.tensor(np.concatenate([t, t]), dtype=torch.long)

print("edge_index shape:", edge_index.shape)
print("edge_attr shape:", edge_attr.shape)

In [ ]:
# ==============================
# 11. 过程级 seed mask（可选但很有用）
# ==============================
process_seed_masks = {}
for p in process_names:
    m = torch.zeros(num_nodes, dtype=torch.bool)
    for idx in process_to_seed_idx[p]:
        m[idx] = True
    process_seed_masks[p] = m


In [ ]:
# ==============================
# 12. 构建 PyG Data 对象
# ==============================
data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr,
    edge_weight=edge_weight,
    y=y
)

# 训练时真正应该用这个
data.supervised_mask = supervised_mask

# 仅为兼容你旧代码里用 train_mask 的写法
data.train_mask = supervised_mask.clone()

# 这些不是 train/val/test 切分，只是辅助信息
data.all_core_mask = all_core_mask
data.hard_seed_mask = hard_seed_mask
data.ambiguous_core_mask = ambiguous_core_mask
data.seed_weight = seed_weight
data.edge_type = edge_type

data.seed_mask_Biosynthesis = process_seed_masks["Biosynthesis"]
data.seed_mask_Esterification = process_seed_masks["Esterification"]
data.seed_mask_Excretion = process_seed_masks["Excretion"]
data.seed_mask_Uptake = process_seed_masks["Uptake"]

print(data)


In [ ]:
# ==============================
# 13. 保存
# ==============================
aux_info = {
    "gene2idx": gene2idx,
    "idx2gene": idx2gene,
    "process_map": process_map,
    "feature_cols": feature_cols,
    "process_to_seed_genes": process_to_seed_genes,
    "note": (
        "No train/val/test split. "
        "all_core_mask = all core genes in graph; "
        "supervised_mask = single-label hard seeds used for CE loss; "
        "ambiguous core genes are included in process seed sets for graph features."
    )
}

torch.save({
    "data": data,
    "node_df": node_df,
    "edge_df": edge_df,
    "hard_seed_df": hard_seed_df,
    "ambiguous_seed_df": ambiguous_seed_df,
    "core_all": core_all,
    "aux_info": aux_info
}, OUT_PYG_FILE)

print("saved to:", OUT_PYG_FILE)